# Track C: Document Q&A with Fact-Check - STARTER
## Lightweight Alternative (Rate-Limit Friendly)

**Only 3 agents | 3 LLM calls per question | ~$0.01-0.03 per run**

| Project | Agents | LLM Calls | Cost/Run |
|---------|--------|-----------|----------|
| Track A: Intelligence Hub | 6 agents | 6 calls | $0.08-0.15 |
| Track B: Debate System | 7 agents | 5-9 calls | $0.10-0.20 |
| **Track C: Q&A + Fact-Check** | **3 agents** | **3 calls** | **$0.01-0.03** |

**Agents:**
1. **Planner** (cheap LLM) - Analyze question, generate search queries
2. **Answerer** (strong LLM) - Generate answer from retrieved chunks
3. **Verifier** (cheap LLM) - Fact-check answer against source

**What's given:** Agent functions (they work individually).  
**Your job:** Build the ENGINEERING that makes them work together.

---

## 1. Setup - Install Dependencies & Set API Key

In [ ]:
# Install required packages
!pip install -q google-genai pymupdf faiss-cpu pydantic python-dotenv numpy

In [ ]:
# Set your Gemini API key
# Get a free key at: https://aistudio.google.com/apikey
import os
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Go to the key icon in left sidebar -> Add secret named GEMINI_API_KEY
try:
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('API key loaded from Colab Secrets')
except:
    # Option 2: Paste directly (less secure)
    os.environ['GEMINI_API_KEY'] = 'YOUR_KEY_HERE'  # <-- Replace with your key
    print('API key set manually')

## 2. helpers.py - Shared Building Blocks (GIVEN)

Everything you need is here:
- `load_and_chunk(pdf_path)` - PDF to text chunks
- `build_index(chunks)` - FAISS vector index
- `search(index, chunks, query)` - Semantic search
- `call_llm_cheap()` / `call_llm_strong()` - LLM calls with retry
- `parse_json(text)` - Safe JSON extraction
- `CostTracker` - Token/cost tracking
- `EvalHarness` - Pipeline benchmarking

In [ ]:
# Suppress noisy gRPC/ALTS warnings
import warnings
warnings.filterwarnings("ignore", message=".*ALTS.*")
warnings.filterwarnings("ignore", category=FutureWarning)
import logging
logging.getLogger("grpc").setLevel(logging.ERROR)

import fitz  # PyMuPDF
import faiss
import numpy as np
import json
import time
import re
import os
import hashlib
from google import genai

# Configure Gemini (new SDK)
_client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
PROVIDER = 'gemini'
print('[helpers] Using Google Gemini API')

# ============================================================
# COST TRACKING
# ============================================================

MODEL_PRICING = {
    'gemini-2.0-flash':       {'input': 0.10,  'output': 0.40},
    'gemini-2.5-flash':       {'input': 0.15,  'output': 0.60},
    'gemini-2.5-pro':         {'input': 1.25,  'output': 10.00},
    'gemini-embedding-001':   {'input': 0.00,  'output': 0.00},
}


class CostTracker:
    """Tracks token usage and estimated cost across all LLM calls."""

    def __init__(self, budget=1.00):
        self.budget = budget
        self.calls = []
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_cost = 0.0

    def record(self, llm_result, agent_name='unknown'):
        tokens = llm_result.get('tokens', {})
        model = llm_result.get('model', 'gemini-2.0-flash')
        inp = tokens.get('input', 0)
        out = tokens.get('output', 0)
        pricing = MODEL_PRICING.get(model, {'input': 1.0, 'output': 3.0})
        cost = (inp * pricing['input'] + out * pricing['output']) / 1_000_000
        self.calls.append({'agent': agent_name, 'model': model,
                           'input_tokens': inp, 'output_tokens': out,
                           'cost': cost, 'latency_ms': llm_result.get('latency_ms', 0)})
        self.total_input_tokens += inp
        self.total_output_tokens += out
        self.total_cost += cost
        return cost

    def remaining(self):
        return max(0, self.budget - self.total_cost)

    def check_budget(self):
        if self.total_cost > self.budget:
            raise RuntimeError(f'Budget exceeded: ${self.total_cost:.4f} of ${self.budget:.2f}')

    def report(self):
        print('\n' + '=' * 65)
        print('  COST REPORT')
        print('=' * 65)
        agent_costs = {}
        for call in self.calls:
            name = call['agent']
            if name not in agent_costs:
                agent_costs[name] = {'calls': 0, 'tokens': 0, 'cost': 0, 'latency': 0}
            agent_costs[name]['calls'] += 1
            agent_costs[name]['tokens'] += call['input_tokens'] + call['output_tokens']
            agent_costs[name]['cost'] += call['cost']
            agent_costs[name]['latency'] += call['latency_ms']
        print(f"\n  {'Agent':<22} {'Calls':>5} {'Tokens':>8} {'Cost':>9} {'Latency':>9}")
        print(f"  {'-' * 57}")
        for name, data in sorted(agent_costs.items(), key=lambda x: -x[1]['cost']):
            print(f"  {name:<22} {data['calls']:>5} {data['tokens']:>8} ${data['cost']:>7.4f} {data['latency']:>7}ms")
        print(f"  {'-' * 57}")
        print(f"  {'TOTAL':<22} {len(self.calls):>5} "
              f"{self.total_input_tokens + self.total_output_tokens:>8} "
              f"${self.total_cost:>7.4f} "
              f"{sum(c['latency_ms'] for c in self.calls):>7}ms")
        print(f"\n  Budget: ${self.budget:.2f} | Spent: ${self.total_cost:.4f} | "
              f"Remaining: ${self.remaining():.4f}")
        print('=' * 65)


# ============================================================
# DOCUMENT PROCESSING
# ============================================================

def load_and_chunk(pdf_path, chunk_size=300, overlap=50):
    """Load a PDF and split into overlapping text chunks."""
    doc = fitz.open(pdf_path)
    pages = len(doc)
    text = ' '.join([page.get_text() for page in doc])
    doc.close()
    words = text.split()
    if not words:
        raise ValueError(f'No text found in {pdf_path}.')
    chunks = []
    step = max(1, chunk_size - overlap)
    for i in range(0, len(words), step):
        chunk = ' '.join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
    print(f'  Loaded {pages} pages, {len(words)} words -> {len(chunks)} chunks '
          f'(size={chunk_size}, overlap={overlap})')
    return chunks


# ============================================================
# EMBEDDINGS
# ============================================================

def embed(text):
    """Convert text into a vector embedding using Gemini."""
    result = _client.models.embed_content(
        model='gemini-embedding-001',
        contents=text,
        config={'task_type': 'RETRIEVAL_DOCUMENT'}
    )
    return np.array(result.embeddings[0].values, dtype='float32')


def build_index(chunks):
    """Build a FAISS index from text chunks."""
    print(f'  Embedding {len(chunks)} chunks...')
    vecs = []
    for i, chunk in enumerate(chunks):
        vecs.append(embed(chunk))
        if (i + 1) % 5 == 0:
            time.sleep(0.5)
        if (i + 1) % 20 == 0 or (i + 1) == len(chunks):
            print(f'    {i + 1}/{len(chunks)} embedded')
    matrix = np.array(vecs).astype('float32')
    index = faiss.IndexFlatL2(matrix.shape[1])
    index.add(matrix)
    print(f'  Index ready: {index.ntotal} vectors, {matrix.shape[1]} dimensions')
    return index


def search(index, chunks, query, k=5):
    """Search FAISS index for the k most similar chunks to query."""
    query_vec = embed(query).reshape(1, -1)
    distances, indices = index.search(query_vec, min(k, index.ntotal))
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if 0 <= idx < len(chunks):
            results.append({'text': chunks[idx], 'score': round(float(1 / (1 + dist)), 4),
                            'index': int(idx)})
    return results


# ============================================================
# LLM CALLING
# ============================================================

def call_llm(prompt, system='You are a helpful assistant.',
             model=None, temperature=0, max_tokens=2000, json_output=False,
             retries=2, backoff_base=2.0):
    """Call the Gemini LLM with automatic retry on transient failures."""
    last_error = None
    for attempt in range(retries + 1):
        try:
            start = time.time()
            m = model or 'gemini-2.0-flash'
            combined_prompt = f'{system}\n\n{prompt}'
            if json_output:
                combined_prompt += '\n\nIMPORTANT: Return ONLY valid JSON. No explanation, no markdown code fences.'
            gen_config = {'temperature': temperature, 'max_output_tokens': max_tokens}
            if json_output:
                gen_config['response_mime_type'] = 'application/json'
            response = _client.models.generate_content(
                model=m, contents=combined_prompt, config=gen_config)
            elapsed = time.time() - start
            usage = response.usage_metadata
            input_tokens = getattr(usage, 'prompt_token_count', 0) if usage else 0
            output_tokens = getattr(usage, 'candidates_token_count', 0) if usage else 0
            return {'text': response.text,
                    'tokens': {'input': input_tokens, 'output': output_tokens},
                    'latency_ms': int(elapsed * 1000), 'model': m}
        except Exception as e:
            last_error = e
            if attempt < retries:
                wait = backoff_base ** attempt
                print(f'    [retry] Attempt {attempt + 1} failed: {e}. Waiting {wait:.0f}s...')
                time.sleep(wait)
            else:
                raise last_error


def call_llm_cheap(prompt, system='You are a helpful assistant.',
                   temperature=0, max_tokens=1000, json_output=False):
    """Cheap/fast model: Gemini 2.0 Flash."""
    return call_llm(prompt, system, model='gemini-2.0-flash',
                    temperature=temperature, max_tokens=max_tokens, json_output=json_output)


def call_llm_strong(prompt, system='You are a helpful assistant.',
                    temperature=0, max_tokens=8192, json_output=False):
    """Strong model: Gemini 2.5 Flash."""
    return call_llm(prompt, system, model='gemini-2.5-flash',
                    temperature=temperature, max_tokens=max_tokens, json_output=json_output)


# ============================================================
# JSON PARSING
# ============================================================

def parse_json(text):
    """Safely parse JSON from LLM output."""
    text = re.sub(r'^```(?:json)?\s*', '', text.strip())
    text = re.sub(r'\s*```$', '', text.strip())
    def _wrap_if_list(parsed):
        if not isinstance(parsed, list): return parsed
        if not parsed or not isinstance(parsed[0], dict): return {'items': parsed}
        first = parsed[0]
        if 'fact' in first: return {'facts': parsed, 'total_extracted': len(parsed)}
        if 'question' in first: return {'questions': parsed}
        return {'items': parsed}
    try: return _wrap_if_list(json.loads(text))
    except json.JSONDecodeError: pass
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try: return _wrap_if_list(json.loads(match.group()))
        except json.JSONDecodeError: pass
    match = re.search(r'\[.*\]', text, re.DOTALL)
    if match:
        try: return _wrap_if_list(json.loads(match.group()))
        except json.JSONDecodeError: pass
    raise ValueError(f'Could not parse JSON from LLM output:\n{text[:300]}...')


# ============================================================
# EVALUATION HARNESS
# ============================================================

class EvalHarness:
    """Benchmark your pipeline against test cases."""
    def __init__(self):
        self.test_cases = []
        self.results = []

    def add_test(self, question, expected_keywords=None, expected_answer=None, difficulty='medium'):
        self.test_cases.append({'question': question, 'expected_keywords': expected_keywords or [],
                                'expected_answer': expected_answer, 'difficulty': difficulty})

    def run(self, pipeline_fn):
        self.results = []
        print(f'\n  Running {len(self.test_cases)} evaluation tests...')
        for i, test in enumerate(self.test_cases, 1):
            start = time.time()
            try:
                result = pipeline_fn(test['question'])
                elapsed = time.time() - start
                answer = result.get('answer', '')
                critic_score = result.get('critic_score', 0)
                answer_lower = answer.lower() if isinstance(answer, str) else str(answer).lower()
                keywords_found = sum(1 for kw in test['expected_keywords'] if kw.lower() in answer_lower)
                keyword_score = keywords_found / len(test['expected_keywords']) if test['expected_keywords'] else 1.0
                composite = keyword_score * 0.5 + critic_score * 0.5
                self.results.append({'question': test['question'], 'difficulty': test['difficulty'],
                                     'keyword_score': round(keyword_score, 2), 'critic_score': round(critic_score, 2),
                                     'composite_score': round(composite, 2), 'latency_s': round(elapsed, 1),
                                     'status': 'pass' if composite >= 0.6 else 'fail'})
            except Exception as e:
                self.results.append({'question': test['question'], 'status': 'error', 'error': str(e),
                                     'composite_score': 0, 'latency_s': round(time.time() - start, 1)})
            s = self.results[-1]
            icon = 'PASS' if s['status'] == 'pass' else 'FAIL' if s['status'] == 'fail' else 'ERR'
            print(f"  {i}. [{icon}] {test['question'][:50]}... (score: {s['composite_score']})")
        return self.results

    def report(self):
        if not self.results: return
        passed = sum(1 for r in self.results if r['status'] == 'pass')
        total = len(self.results)
        avg = sum(r['composite_score'] for r in self.results) / total
        print(f'\n  Results: {passed}/{total} passed ({passed/total*100:.0f}%), Avg score: {avg:.2f}')


# ============================================================
# STATE MANAGEMENT
# ============================================================

def init_state(query=''):
    return {'query': query, 'chunks': [], 'log': [], 'errors': [], 'start_time': time.time()}

def log_agent(state, agent_name, input_summary, output_summary, meta=None):
    entry = {'agent': agent_name, 'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ'),
             'input': input_summary, 'output': output_summary}
    if meta: entry.update(meta)
    state['log'].append(entry)
    return state

def print_log(state):
    print('\n' + '=' * 65)
    print('  AGENT EXECUTION LOG')
    print('=' * 65)
    for entry in state['log']:
        print(f"\n  [{entry['agent'].upper()}] @ {entry.get('timestamp', '?')}")
        if 'tokens' in entry:
            t = entry['tokens']
            print(f"    Tokens: {t.get('input', 0)} in / {t.get('output', 0)} out")
        if 'latency_ms' in entry:
            print(f"    Latency: {entry['latency_ms']}ms")
        output = entry.get('output', '')
        if isinstance(output, str):
            print(f"    Output: {output[:120]}")
    elapsed = time.time() - state.get('start_time', time.time())
    print(f"\n  Total time: {elapsed:.1f}s | Agents called: {len(state['log'])}")
    print('=' * 65)

print('helpers loaded successfully!')

## 3. Test helpers - Verify Everything Works

In [ ]:
print('1. Testing LLM call...')
result = call_llm_cheap("Say 'Hello Workshop!' and nothing else.")
print(f'   Response: {result["text"]}')

print('\n2. Testing CostTracker...')
tracker = CostTracker(budget=0.50)
tracker.record(result, agent_name='test')
tracker.report()

print('\n3. Testing JSON output + parse...')
result = call_llm_cheap('Return JSON: {"status": "ok"}', json_output=True)
parsed = parse_json(result['text'])
print(f'   Parsed: {parsed}')

print('\n4. Testing embedding...')
vec = embed('machine learning is great')
print(f'   Vector shape: {vec.shape}')

print('\nAll tests passed! Ready for Track C.')

## 4. Create a Test PDF

Upload your own PDF, or use this auto-generated test document:

In [ ]:
# Create a test PDF document
doc = fitz.open()
page = doc.new_page()
text = """Machine Learning: A Comprehensive Overview

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:

1. Supervised Learning: The algorithm learns from labeled training data, making predictions or
decisions based on past examples. Common algorithms include Linear Regression, Decision Trees,
Random Forests, and Support Vector Machines.

2. Unsupervised Learning: The algorithm finds patterns in unlabeled data without predefined
categories. K-Means Clustering, PCA, and DBSCAN are popular unsupervised methods.

3. Reinforcement Learning: An agent learns to make decisions by performing actions and receiving
rewards or penalties. Used in robotics, game playing, and autonomous vehicles.

Key Concepts:
- Feature Engineering: Selecting and transforming variables for model input.
- Overfitting: When a model learns noise instead of the underlying pattern.
- Cross-Validation: A technique for assessing model performance on unseen data.
- Bias-Variance Tradeoff: Balancing model complexity and generalization.
- Gradient Descent: An optimization algorithm used to minimize the loss function.

Applications:
Machine learning powers recommendation systems (Netflix, Spotify), natural language processing
(ChatGPT, Google Translate), computer vision (autonomous driving, medical imaging), fraud
detection in banking, and predictive maintenance in manufacturing.

The field continues to evolve rapidly with deep learning, transformer architectures, and
large language models pushing the boundaries of what AI can achieve.

Evaluation Metrics:
- Accuracy: Percentage of correct predictions.
- Precision: Of all positive predictions, how many were actually positive.
- Recall: Of all actual positives, how many were correctly identified.
- F1 Score: Harmonic mean of precision and recall.
- AUC-ROC: Area under the receiver operating characteristic curve.

Challenges:
Data quality, interpretability, computational costs, ethical considerations, and the need for
domain expertise remain significant challenges in practical ML deployments.
"""
page.insert_text((72, 72), text, fontsize=10)
doc.save('test_doc.pdf')
doc.close()
print('test_doc.pdf created!')

# Or upload your own PDF:
# from google.colab import files
# uploaded = files.upload()
# PDF_PATH = list(uploaded.keys())[0]

PDF_PATH = 'test_doc.pdf'

---
# Track C: Document Q&A with Fact-Check

**Architecture:**
```
Question -> Planner -> RETRIEVE (free!) -> Answerer -> Verifier -> Report
               ^                                          |
               +---- retry (verifier feedback) -----------+
```

**Only 3 LLM calls per question!**
- Planner: 1 cheap call
- Answerer: 1 strong call
- Verifier: 1 cheap call
- RETRIEVE step is FREE (vector search only)

**5 Milestones:**
1. Wire sequential pipeline (Planner -> Retrieve -> Answerer -> Verifier)
2. Multi-question mode (reuse FAISS index)
3. Cost tracking + budget enforcement
4. Confidence-based retry (Verifier feedback loops to Planner)
5. Evaluation harness (5 questions = only 15 API calls!)

## Track C - Agent Functions (GIVEN)

These agent functions are complete and work individually. **Don't modify them.**  
Your job is to wire them together into a pipeline in the milestones below.

In [ ]:
# ============================================================
# AGENT PROMPTS (GIVEN)
# ============================================================

PLANNER_SYSTEM = """You are a Question Planner agent.
Analyze the user's question and generate 3 targeted search queries
to find the most relevant information in the document.

Return JSON:
{
    "question_type": "factual | analytical | comparative | opinion",
    "search_queries": ["query1", "query2", "query3"],
    "what_to_look_for": "one sentence describing what a good answer needs"
}"""

ANSWERER_SYSTEM = """You are an Answerer agent. Generate a clear, accurate answer
using ONLY the provided document chunks. Do not use any prior knowledge.

Rules:
- Every claim must be traceable to a chunk
- If information is insufficient, say so explicitly
- Be concise but thorough (3-5 sentences)
- Include specific details, numbers, or quotes when available

Return JSON:
{
    "answer": "Your 3-5 sentence answer here",
    "confidence": "high | medium | low",
    "key_evidence": ["evidence1 from chunks", "evidence2 from chunks"],
    "limitations": "what the document doesn't cover (if any)"
}"""

VERIFIER_SYSTEM = """You are a Verifier agent (fact-checker). Check the answer
against the source chunks. Be STRICT - only confirm claims that are directly
supported by the chunks.

Score each dimension 0-10:
- Accuracy: Are claims supported by the chunks?
- Completeness: Does the answer cover the key info in the chunks?
- Faithfulness: Does the answer avoid adding unsupported info?

Return JSON:
{
    "accuracy_score": 8,
    "completeness_score": 7,
    "faithfulness_score": 9,
    "overall_score": 0.80,
    "verdict": "pass | fail",
    "issues": ["any specific problems found"],
    "suggestion": "one sentence on how to improve (if needed)"
}"""


# ============================================================
# AGENT FUNCTIONS (GIVEN - these work individually)
# ============================================================

def planner(state, tracker):
    """Analyzes question and generates search queries. Works standalone."""
    retry_hint = ''
    if state.get('_verifier_feedback'):
        retry_hint = (f"\n\nPrevious answer was weak. Verifier said: {state['_verifier_feedback']}. "
                      f"Generate DIFFERENT search queries to find better evidence.")
    result = call_llm_cheap(
        system=PLANNER_SYSTEM,
        prompt=f"Question: {state['question']}{retry_hint}\n\nGenerate search queries.",
        json_output=True)
    tracker.record(result, 'planner')
    state['plan'] = parse_json(result['text'])
    log_agent(state, 'planner', {'question': state['question']}, state['plan'],
              {'tokens': result['tokens'], 'latency_ms': result['latency_ms']})
    print(f"  [Planner] Type: {state['plan'].get('question_type', '?')} | "
          f"Queries: {len(state['plan'].get('search_queries', []))}")
    return state


def answerer(state, tracker):
    """Generates answer from retrieved chunks. Works standalone."""
    chunks_text = '\n---\n'.join([c['text'] for c in state.get('retrieved_chunks', [])])
    plan = state.get('plan', {})
    result = call_llm_strong(
        system=ANSWERER_SYSTEM,
        prompt=f"Question: {state['question']}\n"
               f"What to look for: {plan.get('what_to_look_for', 'relevant information')}\n\n"
               f"Document chunks:\n{chunks_text}",
        json_output=True)
    tracker.record(result, 'answerer')
    state['answer'] = parse_json(result['text'])
    log_agent(state, 'answerer', {'chunks': len(state.get('retrieved_chunks', []))},
              state['answer'].get('answer', '')[:80],
              {'tokens': result['tokens'], 'latency_ms': result['latency_ms']})
    print(f"  [Answerer] Confidence: {state['answer'].get('confidence', '?')} | "
          f"Evidence: {len(state['answer'].get('key_evidence', []))} pieces")
    return state


def verifier(state, tracker):
    """Fact-checks the answer against source chunks. Works standalone."""
    chunks_text = '\n---\n'.join([c['text'] for c in state.get('retrieved_chunks', [])[:8]])
    answer_data = state.get('answer', {})
    result = call_llm_cheap(
        system=VERIFIER_SYSTEM,
        prompt=f"Question: {state['question']}\n\n"
               f"Answer to verify:\n{json.dumps(answer_data, indent=2)}\n\n"
               f"Source chunks:\n{chunks_text}",
        json_output=True)
    tracker.record(result, 'verifier')
    parsed = parse_json(result['text'])
    state['verification'] = parsed
    state['verifier_score'] = parsed.get('overall_score', 0)
    state['_verifier_feedback'] = parsed.get('suggestion', '')
    log_agent(state, 'verifier', {'answer_length': len(answer_data.get('answer', ''))},
              f"Score={state['verifier_score']}, {parsed.get('verdict', '?')}",
              {'tokens': result['tokens'], 'latency_ms': result['latency_ms']})
    print(f"  [Verifier] Score: {state['verifier_score']}/1.0 - "
          f"{parsed.get('verdict', '?').upper()}")
    return state


print('Track C agents loaded! (3 agents: Planner, Answerer, Verifier)')

## Milestone 1: Wire the Sequential Pipeline

**Your task:** Call the 3 agents in the correct order with proper state flow.

**Pipeline:**
```
Load PDF -> Build Index -> Planner -> RETRIEVE (free!) -> Answerer -> Verifier -> Report
```

**Key difference from Track A/B:** The RETRIEVE step between Planner and Answerer uses **no LLM calls** - it's pure vector search!

**Hints:**
- `chunks = load_and_chunk(pdf_path)` returns raw text chunks
- `index = build_index(chunks)` creates FAISS index
- `search(index, chunks, query, k=5)` returns `[{text, score, index}]`
- Planner gives you queries in `state['plan']['search_queries']`
- Loop through queries, call `search()` for each, combine + deduplicate
- Store results in `state['retrieved_chunks']`

In [ ]:
def run_qa(pdf_path, question, budget=0.30):
    """Run the Q&A pipeline: Planner -> Retrieve -> Answerer -> Verifier."""
    tracker = CostTracker(budget=budget)
    print(f"\n{'=' * 60}\n  DOCUMENT Q&A WITH FACT-CHECK\n{'=' * 60}")
    print(f'  Question: {question}\n  Budget: ${budget:.2f}')

    # Step 1: Load document + build FAISS index
    print('\n[1] Loading document...')
    # YOUR CODE HERE
    # HINT:
    #   chunks = load_and_chunk(pdf_path)
    #   index = build_index(chunks)


    # Step 2: Init state with question
    # YOUR CODE HERE
    # HINT:
    #   state = init_state(query=question)
    #   state['question'] = question
    #   state['_index'] = index
    #   state['_all_chunks'] = chunks


    # Step 3: Run Planner (1st LLM call - cheap)
    print('\n[2] Planning search strategy...')
    # YOUR CODE HERE
    # HINT: state = planner(state, tracker)


    # Step 4: RETRIEVE - use planner's queries to find chunks (FREE - no LLM!)
    print('\n[3] Retrieving evidence...')
    # YOUR CODE HERE
    # HINT:
    #   all_results = []
    #   for q in state['plan']['search_queries']:
    #       all_results.extend(search(index, chunks, q, k=5))
    #   # Deduplicate by chunk index
    #   seen = set()
    #   unique = [c for c in all_results if c['index'] not in seen and not seen.add(c['index'])]
    #   state['retrieved_chunks'] = unique[:12]
    #   print(f'  Retrieved {len(state["retrieved_chunks"])} unique chunks')


    # Step 5: Run Answerer (2nd LLM call - strong)
    print('\n[4] Generating answer...')
    # YOUR CODE HERE
    # HINT: state = answerer(state, tracker)


    # Step 6: Run Verifier (3rd LLM call - cheap)
    print('\n[5] Fact-checking...')
    # YOUR CODE HERE
    # HINT: state = verifier(state, tracker)


    # Step 7: Print results
    print('\n[6] Results:')
    # YOUR CODE HERE
    # HINT:
    #   print(f"  Answer: {state['answer']['answer']}")
    #   print(f"  Confidence: {state['answer']['confidence']}")
    #   print(f"  Verifier: {state['verifier_score']}/1.0 - {state['verification']['verdict'].upper()}")
    #   for i, ev in enumerate(state['answer'].get('key_evidence', []), 1):
    #       print(f"    Evidence {i}: {ev[:80]}")


    print_log(state)
    tracker.report()
    return state

# Uncomment when ready:
# qa_state = run_qa(PDF_PATH, 'What are the main types of machine learning?')

In [ ]:
# TEST MILESTONE 1 - Try different questions!

# qa_state = run_qa(PDF_PATH, 'What are the main types of machine learning?')
# qa_state = run_qa(PDF_PATH, 'What evaluation metrics are used in ML?')
# qa_state = run_qa(PDF_PATH, 'What are the challenges in machine learning?')

## Milestone 2: Multi-Question Mode

**Your task:** Ask multiple questions about the SAME document without rebuilding the FAISS index each time.

**Why?** Building the index is the slowest part (embedding all chunks). Reusing it saves time AND embedding API calls.

**Requirements:**
1. Load document and build index **ONCE**
2. Loop through questions, running planner -> retrieve -> answerer -> verifier for each
3. Collect results and print a summary table

**Hint:** Factor out the load+index step, pass `index` and `chunks` to a `run_single_qa()` function.

In [ ]:
def run_single_qa(question, index, chunks, tracker):
    """
    YOUR CODE: Run Q&A pipeline for one question, reusing existing index.
    Returns state dict with answer and verification.
    """
    # YOUR CODE HERE
    # HINT: Same as run_qa steps 2-7, but skip loading/indexing
    #   state = init_state(query=question)
    #   state['question'] = question
    #   state['_index'] = index
    #   state['_all_chunks'] = chunks
    #   state = planner(state, tracker)
    #   ... retrieve chunks ...
    #   state = answerer(state, tracker)
    #   state = verifier(state, tracker)
    #   return state
    pass


def run_multi_qa(pdf_path, questions, budget=0.50):
    """
    YOUR CODE: Run multiple questions on same document.
    Build index ONCE, reuse for all questions.
    """
    # YOUR CODE HERE
    # HINT:
    #   tracker = CostTracker(budget=budget)
    #   chunks = load_and_chunk(pdf_path)
    #   index = build_index(chunks)
    #
    #   results = []
    #   for i, question in enumerate(questions, 1):
    #       print(f'\n--- Question {i}/{len(questions)}: {question} ---')
    #       state = run_single_qa(question, index, chunks, tracker)
    #       results.append({
    #           'question': question,
    #           'answer': state['answer']['answer'],
    #           'confidence': state['answer']['confidence'],
    #           'score': state['verifier_score']
    #       })
    #
    #   # Print summary table
    #   print(f"\n{'=' * 60}")
    #   print('  MULTI-QUESTION SUMMARY')
    #   print(f"{'=' * 60}")
    #   for i, r in enumerate(results, 1):
    #       print(f"  {i}. [{r['confidence'].upper()}] Score: {r['score']}/1.0")
    #       print(f"     Q: {r['question']}")
    #       print(f"     A: {r['answer'][:100]}...")
    #   tracker.report()
    pass

# Uncomment when ready:
# run_multi_qa(PDF_PATH, [
#     'What are the main types of machine learning?',
#     'What evaluation metrics are used in ML?',
#     'What are the challenges in practical ML deployments?'
# ])

## Milestone 3: Cost Tracking + Budget Enforcement

**Your task:** Add `tracker.check_budget()` between pipeline stages.

**Why it matters:** With only 3 agents, this pipeline is very cheap (~$0.01-0.03 per question). But in multi-question or retry mode, costs add up.

**Requirements:**
1. Add `tracker.check_budget()` after each agent call
2. Wrap in `try/except RuntimeError` for clean budget-exceeded message
3. Test with `budget=0.005` to see it stop gracefully

**Engineering question:** Where should budget checks go?
- After every agent? (safest)
- After each pipeline stage? (practical)
- Only at the end? (too late!)

In [ ]:
# YOUR CODE HERE
# Go back to run_qa() or run_single_qa() and add budget checks:
#
# After planner:
#   try:
#       tracker.check_budget()
#   except RuntimeError as e:
#       print(f'  Budget exceeded after planner: {e}')
#       return state  # Return partial results
#
# After answerer:
#   tracker.check_budget()  # Same pattern
#
# After verifier:
#   tracker.check_budget()  # Same pattern

# Test with tiny budget:
# qa_state = run_qa(PDF_PATH, 'What is machine learning?', budget=0.005)

## Milestone 4: Confidence-Based Retry

**Your task:** When the Verifier scores < 0.7, feed feedback back to Planner and retry.

**How it works:**
1. After Verifier, check `state['verifier_score']`
2. If score < 0.7 and retries remaining:
   - `state['_verifier_feedback']` already has the suggestion
   - The `planner()` function already reads this feedback (check its code!)
   - Re-run: Planner (with feedback) -> Retrieve -> Answerer -> Verifier
3. Each retry = only 3 more API calls (~$0.01-0.03)

**When NOT to retry:**
- Score is 0.0 (question unanswerable from document)
- Already retried `max_retries` times
- Budget too low for another cycle

In [ ]:
def run_qa_with_retry(pdf_path, question, budget=0.30, max_retries=2):
    """
    YOUR CODE: Q&A pipeline with confidence-based retry.
    If verifier score < 0.7, feed feedback to planner and retry.
    """
    # YOUR CODE HERE
    # HINT: Start with your run_qa code, then add retry loop:
    #
    #   tracker = CostTracker(budget=budget)
    #   chunks = load_and_chunk(pdf_path)
    #   index = build_index(chunks)
    #   state = init_state(query=question)
    #   state['question'] = question
    #   state['_index'] = index
    #   state['_all_chunks'] = chunks
    #   state['retry_count'] = 0
    #
    #   for attempt in range(max_retries + 1):
    #       print(f'\n--- Attempt {attempt + 1} ---')
    #       state = planner(state, tracker)
    #       # ... retrieve chunks ...
    #       state = answerer(state, tracker)
    #       state = verifier(state, tracker)
    #
    #       if state['verifier_score'] >= 0.7:
    #           print(f'  PASSED on attempt {attempt + 1}!')
    #           break
    #       elif attempt < max_retries:
    #           print(f'  Score {state["verifier_score"]} < 0.7 - retrying...')
    #           print(f'  Feedback: {state["_verifier_feedback"]}')
    #           state['retry_count'] += 1
    #           # Budget check before retry
    #           try:
    #               tracker.check_budget()
    #           except RuntimeError:
    #               print('  Budget too low for retry. Stopping.')
    #               break
    #       else:
    #           print(f'  Max retries reached. Final score: {state["verifier_score"]}')
    #
    #   tracker.report()
    #   return state
    pass

# Uncomment when ready:
# qa_state = run_qa_with_retry(PDF_PATH, 'What is the relationship between bias and variance in ML?')

## Milestone 5: Evaluation Harness

**Your task:** Test your pipeline on 5 questions and measure quality.

**Why this is great for Track C:**
- 5 questions x 3 LLM calls = **only 15 API calls total**
- Compare: Track A eval = 5 x 6 = 30 calls, Track B = even more

**Requirements:**
1. Create an `EvalHarness` with 5 test cases (question + expected_keywords)
2. Write `pipeline_for_eval(question)` that reuses cached index
3. Return `{'answer': ..., 'critic_score': state['verifier_score']}`
4. Target: >60% pass rate

In [ ]:
def run_evaluation(pdf_path):
    """
    YOUR CODE: Build and run evaluation harness.
    """
    # YOUR CODE HERE
    # HINT:
    #   # Step 1: Load + index ONCE
    #   chunks = load_and_chunk(pdf_path)
    #   index = build_index(chunks)
    #   tracker = CostTracker(budget=1.00)
    #
    #   # Step 2: Define test cases
    #   harness = EvalHarness()
    #   harness.add_test(
    #       question='What are the types of machine learning?',
    #       expected_keywords=['supervised', 'unsupervised', 'reinforcement'],
    #       difficulty='easy')
    #   harness.add_test(
    #       question='What is overfitting?',
    #       expected_keywords=['noise', 'pattern', 'model'],
    #       difficulty='easy')
    #   harness.add_test(
    #       question='What evaluation metrics are available?',
    #       expected_keywords=['accuracy', 'precision', 'recall', 'F1'],
    #       difficulty='medium')
    #   harness.add_test(
    #       question='What are practical applications of ML?',
    #       expected_keywords=['recommendation', 'vision', 'fraud'],
    #       difficulty='medium')
    #   harness.add_test(
    #       question='How does gradient descent relate to the bias-variance tradeoff?',
    #       expected_keywords=['optimization', 'complexity'],
    #       difficulty='hard')
    #
    #   # Step 3: Pipeline function (reuses index!)
    #   def pipeline_for_eval(question):
    #       state = run_single_qa(question, index, chunks, tracker)
    #       return {
    #           'answer': state['answer']['answer'],
    #           'critic_score': state['verifier_score']
    #       }
    #
    #   # Step 4: Run!
    #   harness.run(pipeline_for_eval)
    #   harness.report()
    #   tracker.report()
    pass

# Uncomment when ready:
# run_evaluation(PDF_PATH)

## Report Formatting (GIVEN)

Use this to pretty-print any Q&A result:

In [ ]:
def format_report(state):
    """Format Q&A results. Given - don't modify."""
    a = state.get('answer', {})
    v = state.get('verification', {})
    p = state.get('plan', {})

    r = ['\n' + '=' * 60, '  Q&A REPORT', '=' * 60]
    r.append(f"\n  Question: {state.get('question', '?')}")
    r.append(f"  Type: {p.get('question_type', '?')}")
    r.append(f"\n  Answer: {a.get('answer', '?')}")
    r.append(f"  Confidence: {a.get('confidence', '?')}")
    r.append(f"\n  Evidence:")
    for i, ev in enumerate(a.get('key_evidence', []), 1):
        r.append(f"    {i}. {ev[:100]}")
    if a.get('limitations'):
        r.append(f"\n  Limitations: {a['limitations']}")
    r.append(f"\n  Verification:")
    r.append(f"    Accuracy:     {v.get('accuracy_score', '?')}/10")
    r.append(f"    Completeness: {v.get('completeness_score', '?')}/10")
    r.append(f"    Faithfulness: {v.get('faithfulness_score', '?')}/10")
    r.append(f"    Overall:      {v.get('overall_score', '?')}/1.0 - {v.get('verdict', '?').upper()}")
    if v.get('issues'):
        r.append(f"    Issues: {', '.join(v['issues'])}")
    if state.get('retry_count', 0) > 0:
        r.append(f"\n  Retries: {state['retry_count']}")
    r.append('=' * 60)
    return '\n'.join(r)

# Usage: print(format_report(qa_state))

---
## Try Your Own

Upload your own PDF and ask questions:

In [ ]:
# Upload your own PDF
from google.colab import files
uploaded = files.upload()
YOUR_PDF = list(uploaded.keys())[0]
print(f'Uploaded: {YOUR_PDF}')

In [ ]:
# Ask a single question
# qa_state = run_qa(YOUR_PDF, 'Your question here?')
# print(format_report(qa_state))

In [ ]:
# Ask multiple questions (reuses index!)
# run_multi_qa(YOUR_PDF, [
#     'First question?',
#     'Second question?',
#     'Third question?'
# ])